In [1]:
import cv2
import numpy as np
import glob
import os

# --- 4つの角の座標を正しい順番（左上、右上、右下、左下）に並び替える関数 ---
def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]      # 左上
    rect[2] = pts[np.argmax(s)]      # 右下
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]   # 右上
    rect[3] = pts[np.argmax(diff)]   # 左下
    return rect

print("--- 膜の自動切り抜き＆保存ツール（背景色差分 最強版） ---")
print("※ プログラムと同じフォルダに 'crop_ファイル名' として保存します")
print("-" * 50)

# 実行しているプログラム（.ipynb）と「同じフォルダ」にある画像をすべて取得
all_images = list(set(
    glob.glob("*.jpg") + 
    glob.glob("*.JPG") + 
    glob.glob("*.jpeg") + 
    glob.glob("*.JPEG") +
    glob.glob("*.png") +
    glob.glob("*.PNG")
))

# ★すでに切り抜いた画像（crop_～）を処理対象から外す（二重処理防止）
image_files = [f for f in all_images if not f.startswith("crop_")]
image_files.sort()

# ★何枚見つかったかを表示するチェック機能
print(f"📁 読み込む場所: プログラムと同じフォルダ")
print(f"📷 見つかった画像: {len(image_files)} 枚")

if len(image_files) == 0:
    print("⚠️ エラー: プログラムと同じフォルダに処理対象の画像が1枚も見つかりません！")
    print("処理したい画像を、このプログラム（.ipynb）と同じフォルダの中に移動させてください。")
    print("-" * 50)

for img_path in image_files:
    # 日本語ファイル名等エラー対策でnumpy経由で読み込み
    nparr = np.fromfile(img_path, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    
    if img is None:
        continue

    # --- 1. 画像の「端っこ（暗箱の影など）」を無視する（中央フォーカス） ---
    height, width = img.shape[:2]
    # 探索範囲を絞る（端の影を確実に無視するため、20%〜80%の範囲に限定）
    cx1, cx2 = int(width * 0.2), int(width * 0.8)
    cy1, cy2 = int(height * 0.2), int(height * 0.8)
    center_img = img[cy1:cy2, cx1:cx2]

    # --- 2. 膜を探す（背景色との違いで見つける最強法） ---
    # 四隅の平均色を背景（白い紙）の色として取得
    h_c, w_c = center_img.shape[:2]
    tl = center_img[0:10, 0:10].mean(axis=(0,1))
    tr = center_img[0:10, w_c-10:w_c].mean(axis=(0,1))
    bl = center_img[h_c-10:h_c, 0:10].mean(axis=(0,1))
    br = center_img[h_c-10:h_c, w_c-10:w_c].mean(axis=(0,1))
    bg_color = (tl + tr + bl + br) / 4.0

    # 画像全体と背景色との「色の違い（距離）」を計算
    diff = np.linalg.norm(center_img.astype(np.float32) - bg_color, axis=2)
    
    # 背景の色と大きく違う部分（距離が40以上）を膜として白(255)にする
    _, thresh = cv2.threshold(diff.astype(np.uint8), 40, 255, cv2.THRESH_BINARY)
    
    # ノイズ消し
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        print(f"⚠️ {os.path.basename(img_path):20} | 膜が見つかりませんでした")
        continue

    # 一番面積が大きい輪郭（＝ペロブスカイト膜）を見つける
    largest_contour = max(contours, key=cv2.contourArea)
    
    # --- 3. 傾き補正（射影変換）の処理 ---
    # 傾きを考慮した「最小の四角形」を取得し、4つの角の座標を取り出す
    rect = cv2.minAreaRect(largest_contour)
    box = cv2.boxPoints(rect)
    box = np.float32(box)

    rect_pts = order_points(box)

    # 変換後の画像サイズを決定（ペロブスカイト基板は正方形なので、縦横を同じサイズに強制）
    side_length = int(max(rect[1][0], rect[1][1]))
    
    dst_pts = np.float32([
        [0, 0],
        [side_length - 1, 0],
        [side_length - 1, side_length - 1],
        [0, side_length - 1]
    ])

    # 変換マトリックスを計算し、画像を真っ直ぐに引き伸ばす
    matrix = cv2.getPerspectiveTransform(rect_pts, dst_pts)
    warped_img = cv2.warpPerspective(center_img, matrix, (side_length, side_length))

    # --- 4. マージンカットと保存 ---
    # 端の影や背景を消すために、補正後の画像から少し内側(5ピクセル)を切り抜く
    margin = 5
    if side_length > margin * 2:
        cropped_img = warped_img[margin:side_length-margin, margin:side_length-margin]
    else:
        cropped_img = warped_img

    # ファイル保存（新しくフォルダを作らず、同じ場所に保存）
    filename = os.path.basename(img_path)
    save_path = "crop_" + filename
    
    _, encoded_img = cv2.imencode(".jpg", cropped_img)
    with open(save_path, "wb") as f:
        encoded_img.tofile(f)

    print(f"✔️ {filename:20} -> {save_path} に保存しました")

print("-" * 50)
print("✨ すべての処理が完了しました！")

--- 膜の自動切り抜き＆保存ツール（背景色差分 最強版） ---
※ プログラムと同じフォルダに 'crop_ファイル名' として保存します
--------------------------------------------------
📁 読み込む場所: プログラムと同じフォルダ
📷 見つかった画像: 2 枚
✔️ IBACl0.jpg           -> crop_IBACl0.jpg に保存しました
✔️ IBACl50.jpg          -> crop_IBACl50.jpg に保存しました
--------------------------------------------------
✨ すべての処理が完了しました！


In [2]:
import cv2
import numpy as np
import glob
import os

print("--- 100x100 メッシュ RGB分布解析ツール ---")
print("ファイル名      | 良品色(暗い色)の割合 | Rばらつき | Gばらつき | Bばらつき")
print("-" * 75)

# プログラムと同じフォルダにあるすべての画像を取得
# ★ list(set(...)) を使うことで、Windows環境での「同じ画像の二重読み込み」を防ぎます
all_images = list(set(
    glob.glob("*.jpg") + 
    glob.glob("*.JPG") + 
    glob.glob("*.jpeg") + 
    glob.glob("*.png")
))

# ★解析対象にする画像を指定（今回は "RbCl" から始まる画像）
image_files = [f for f in all_images if os.path.basename(f).startswith("IBACl")]
image_files.sort()

if len(image_files) == 0:
    print("⚠️ エラー: 'IBACl から始まる画像が見つかりません。")
    print("-" * 75)

for img_path in image_files:
    img = cv2.imread(img_path)
    if img is None:
        continue

    # 1. 100x100のメッシュ化（縮小して平均化）
    # cv2.INTER_AREA を使うことで、細かいノイズを潰してマスの平均色を正確に出せます
    mesh_size = 100
    mesh_img = cv2.resize(img, (mesh_size, mesh_size), interpolation=cv2.INTER_AREA)

    # 2. BGR から RGB に変換（扱いやすくするため）
    rgb_img = cv2.cvtColor(mesh_img, cv2.COLOR_BGR2RGB)
    
    # R, G, B それぞれの色ごとにデータを分割
    r, g, b = cv2.split(rgb_img)

    # 3. 各色の「ばらつき（標準偏差）」を計算
    # 膜が均一であればあるほど、この数値は小さくなります
    r_std = np.std(r)
    g_std = np.std(g)
    b_std = np.std(b)

    # 4. 「1万マスに対する特定の色の割合」を計算
    # ペロブスカイトの成功膜は黒〜焦げ茶色（RGBの値がすべて低い）になります。
    # ここでは仮に、RGBすべてが「80以下」のマスを「良品色のマス」と定義して数えます。
    good_pixels = np.sum((r < 80) & (g < 80) & (b < 80))
    
    # 全体（10000マス）に対する良品マスの割合（%）を算出
    total_pixels = mesh_size * mesh_size
    good_ratio = (good_pixels / total_pixels) * 100

    # 結果を出力
    filename = os.path.basename(img_path)
    print(f"{filename:15} | {good_ratio:6.1f}%             | R:{r_std:5.1f} | G:{g_std:5.1f} | B:{b_std:5.1f}")

print("-" * 75)

--- 100x100 メッシュ RGB分布解析ツール ---
ファイル名      | 良品色(暗い色)の割合 | Rばらつき | Gばらつき | Bばらつき
---------------------------------------------------------------------------
IBACl0_cut.jpg  |   28.2%             | R:  9.8 | G:  9.7 | B:  9.5
IBACl50_cut.jpg |   30.3%             | R:  4.1 | G:  4.4 | B:  4.6
---------------------------------------------------------------------------
